In [2]:
# 读取jsonl文件
import json

def read_first_n_lines_jsonl(file_path, n):
    """
    读取并解析一个 JSONL 文件的前 n 行。

    参数:
    file_path (str): JSONL 文件的路径。
    n (int): 需要读取的行数。

    返回:
    list: 一个包含已解析的JSON对象（字典）的列表。
    """
    data = []
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            for i, line in enumerate(f):
                # 如果已经读取了 n 行，则停止循环
                if i >= n:
                    break
                # 解析当前行并添加到列表中
                data.append(json.loads(line.strip()))
    except FileNotFoundError:
        print(f"错误：文件 '{file_path}' 未找到。")
    except json.JSONDecodeError as e:
        # 如果某一行不是有效的JSON，会捕获错误
        print(f"错误：解析第 {i+1} 行时发生JSON解码错误: {e}")
    except Exception as e:
        print(f"发生未知错误: {e}")
        
    return data




# 2. 设置文件路径和要读取的行数
file_path = 'AdaReasoner/AdaDataCurationweb/test_results/guichat/7b_responses_llm.jsonl'
num_lines_to_read = 5

# 3. 调用函数读取文件
first_lines = read_first_n_lines_jsonl(file_path, num_lines_to_read)

# 4. 打印结果
if first_lines:
    print(f"成功读取文件 '{file_path}' 的前 {len(first_lines)} 行内容：")
    for item in first_lines:
        print(item)

成功读取文件 'AdaReasoner/AdaDataCurationweb/test_results/guichat/7b_responses_llm.jsonl' 的前 5 行内容：
{'average_llm_score': 0.7570202110534788}
{'id': 1, 'image_id': '0e750975c41bf199c11c1ea035ecb5cf', 'task_type': 'single-turn-vqa', 'question': 'How does The Fun Experts handle my email privacy?', 'ground_truth': "The Fun Experts assure users of their commitment to email privacy by explicitly stating on their webpage that they never share your data with any third parties. This promise is highlighted in a section of the site that likely aims to reassure visitors when they are considering signing up for the Fun Experts Newsletter. The company's dedication to protecting customer data is a key part of their privacy practices, especially in relation to the collection and use of email addresses. This information is typically presented in close proximity to where users can input their email addresses to subscribe to the newsletter, ensuring that the privacy assurance is seen during the sign-up proces

In [ ]:
import json
import os

# --- 配置 ---
# 输入的 jsonl 文件路径
INPUT_JSONL_FILE = 'tool_server/tf_eval/scripts/logs/ckpt/web/gemini_test_test_format.jsonl'
# 包含图片子文件夹 (web_0, web_1, ...) 的根目录
IMAGES_ROOT_FOLDER = 'tool_server/tf_eval/scripts/logs/ckpt/web/middle_images_test'
# 输出的 jsonl 文件路径
OUTPUT_JSONL_FILE = 'tool_server/tf_eval/scripts/logs/ckpt/web/gemini_test_test_format_conversation.jsonl'
# 每条数据都会使用的系统提示 (System Prompt)
SYSTEM_PROMPT = "You are a helpful assistant that can use tools to answer questions based on images. Please think step by step and call the appropriate tools to find the answer."
# --- 结束配置 ---

def find_value_recursively(key, obj):
    """
    在嵌套的字典或列表中递归搜索一个键 (key) 并返回其值。
    """
    if isinstance(obj, dict):
        if key in obj:
            return obj[key]
        for k, v in obj.items():
            found = find_value_recursively(key, v)
            if found is not None:
                return found
    elif isinstance(obj, list):
        for item in obj:
            found = find_value_recursively(key, item)
            if found is not None:
                return found
    return None

def extract_question_robust(original_question_text):
    """
    更健壮地从 original_question 字段中提取问题部分。
    """
    if not isinstance(original_question_text, str):
        return "ERROR: original_question field is not a string or was not found."
        
    parts = original_question_text.split("Answer_with_box:", 1)
    question_part = parts[0]
    
    if "Question:" in question_part:
        question = question_part.split("Question:", 1)[1]
        return question.strip()
    
    print(f"[INFO] 'Question: ... Answer_with_box:' pattern not found. Using raw text.")
    return original_question_text.strip()


def process_data_final():
    """
    终极版处理函数，使用递归搜索、新的图片添加逻辑、移除 tool_reward，并修正了图片编号逻辑。
    """
    print(f"开始处理文件: {INPUT_JSONL_FILE}")
    
    output_dir = os.path.dirname(OUTPUT_JSONL_FILE)
    if not os.path.exists(output_dir):
        print(f"输出目录 {output_dir} 不存在，正在创建...")
        os.makedirs(output_dir)

    processed_count = 0
    skipped_count = 0

    with open(OUTPUT_JSONL_FILE, 'w', encoding='utf-8') as outfile:
        with open(INPUT_JSONL_FILE, 'r', encoding='utf-8') as infile:
            for line_num, line in enumerate(infile, 1):
                try:
                    data = json.loads(line)
                    
                    idx = find_value_recursively('idx', data)
                    original_question = find_value_recursively('original_question', data)
                    model_responses = find_value_recursively('model_response', data) or []
                    tool_responses = find_value_recursively('tool_response', data) or []
                    if isinstance(idx, list): idx = idx[0]

                    if not idx:
                        print(f"\n[警告] 第 {line_num} 行无法找到 'idx'，已跳过。")
                        skipped_count += 1
                        continue
                    
                    print(f"\n--- 正在处理第 {line_num} 行, idx: {idx} ---")

                    conversation = []
                    conversation.append({"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]})

                    question = extract_question_robust(original_question)
                    initial_image_path = os.path.join(IMAGES_ROOT_FOLDER, idx, 'img_1.png')
                    print(f"提取的问题: '{question}'")
                    conversation.append({
                        "role": "user",
                        "content": [
                            {"type": "text", "text": question},
                            {"type": "image_url", "image_url": {"url": initial_image_path}}
                        ]
                    })
                    
                    print(f"找到 {len(model_responses)} 条模型响应。")
                    print(f"找到 {len(tool_responses)} 条工具响应。")
                    
                    num_tool_calls = len(tool_responses)
                    
                    # --- 核心逻辑修改：初始化图片计数器 ---
                    # img_1 是固定的，所以下一个生成的图片从 img_2 开始
                    image_counter = 2

                    for i in range(num_tool_calls):
                        conversation.append({"role": "assistant", "content": [{"type": "text", "text": model_responses[i]}]})
                        
                        tool_response_obj = tool_responses[i]
                        tool_response_for_display = tool_response_obj.copy()
                        tool_response_for_display.pop('tool_reward', None)
                        tool_response_text = json.dumps(tool_response_for_display, indent=2)
                        
                        user_content = [{"type": "text", "text": tool_response_text}]
                        
                        add_image = (
                            tool_response_obj.get('status') == 'success' and
                            tool_response_obj.get('tool_response_from') == 'Crop' and
                            tool_response_obj.get('tool_reward') == 4
                        )
                        
                        # --- 核心逻辑修改：使用并增加图片计数器 ---
                        if add_image:
                            # 使用当前的计数器值作为图片编号
                            image_path = os.path.join(IMAGES_ROOT_FOLDER, idx, f'img_{image_counter}.png')
                            user_content.append({"type": "image_url", "image_url": {"url": image_path}})
                            print(f"为工具 {tool_response_obj.get('tool_response_from')} 添加了图片 img_{image_counter}.png")
                            # 只有在成功添加图片后，才将计数器加一，为下一张图片做准备
                            image_counter += 1
                        
                        conversation.append({"role": "user", "content": user_content})

                    if len(model_responses) > num_tool_calls:
                        final_answer = model_responses[-1]
                        conversation.append({"role": "assistant", "content": [{"type": "text", "text": final_answer}]})
                        print("已添加最终回答。")
                    else:
                        if num_tool_calls > 0 or len(model_responses) > 0:
                             print("[警告] 未找到独立的最终回答。")

                    outfile.write(json.dumps(conversation, ensure_ascii=False) + '\n')
                    processed_count += 1

                except json.JSONDecodeError:
                    print(f"\n[错误] 第 {line_num} 行JSON格式错误，已跳过。")
                    skipped_count += 1
                except Exception as e:
                    print(f"\n[严重错误] 第 {line_num} 行处理失败: {e}")
                    skipped_count += 1
    
    print(f"\n--- 处理完成 ---")
    print(f"成功处理: {processed_count} 条数据")
    print(f"跳过/失败: {skipped_count} 条数据")
    print(f"结果已保存到: {OUTPUT_JSONL_FILE}")


# --- 运行脚本 ---
if __name__ == '__main__':
    if not os.path.exists(INPUT_JSONL_FILE):
        print(f"错误: 输入文件 '{INPUT_JSONL_FILE}' 不存在。请检查路径。")
    elif not os.path.exists(IMAGES_ROOT_FOLDER):
        print(f"错误: 图片根目录 '{IMAGES_ROOT_FOLDER}' 不存在。请检查路径。")
    else:
        process_data_final()